# Wczytywanie danych do ramki danych pandas

In [12]:
import pandas as pd
from pathlib import Path


# Względna ścieżka do pliku JSON z ofertami pracy
# input_path = Path(input("Podaj względna ściężkę do pliku .json z ofertami pracy"))

# Sztywno zapisana ścieżka do pliku .json, tak aby nie wpisywać ciągle u góry scieżki
input_path = Path("../../scrapping-worker/justjoinit/offers.json")

# Wczytanie danych do ramki danych pandas (na tym etapie dane są "nieczyste")
df: pd.DataFrame = pd.read_json(
    input_path,
    orient="records"
)

# Podgląd pierwszych wierszy DataFrame
df.head(5)

,Details,Tech stack,Salary
0,{},{},Undisclosed salary
1,{},{},Undisclosed salary
2,{},{},Undisclosed salary
3,"{'Type of work': 'Freelance', 'Experience': 'S...","{'AWS': 'master', 'ETL': 'advanced', 'Power BI...",Undisclosed salary
4,{},{},Undisclosed salary


# Rozwijanie zagnieżdzionch słowników

### W niektórych kolumnach, takich jak "Details" czy "Tech Stack", mogą znajdować się zagnieżdżone słowniki 
### Aby uzyskać z nich dodatkowe atrybuty jako osobne kolumny w DataFrame, można je „rozwinąć” za pomocą pd.apply(pd.Series)


In [13]:
def flatten_column(df: pd.DataFrame, col_label: str) -> pd.DataFrame:
    """
    Rozwija słowniki zawarte w kolumnie `col_label` ramki danych `df`
    i zwraca nowy DataFrame z rozwiniętymi kolumnami.
    """
    expanded_df = df[col_label].apply(pd.Series)
    expanded_df.fillna("-", inplace=True)
    return expanded_df

def flatten_multiple_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Rozwija wiele kolumn zawierających słowniki i łączy je z oryginalnym DataFrame.
    """
    flattened_dataframes = []
    for col_label in columns:
        flatten_df = flatten_column(df, col_label)
        flattened_dataframes.append(flatten_df)
    # Łączymy oryginalny DataFrame z rozwiniętymi kolumnami i usuwamy oryginalne kolumny
    return pd.concat([df] + flattened_dataframes, axis=1).drop(columns=columns)

# Podział ramki danych na części tematyczne
# tech_df: wymagania technologiczne
# details_df: szczegóły ofert (zarobki, rodzaj zatrudnienia)
# lang_df: znajomość języków
tech_df: pd.DataFrame = df[["Tech stack"]]
details_df: pd.DataFrame = df[["Details", "Salary"]]

# # Lista kolumn do rozwinięcia dla każdej ramki danych
tech_cols_to_flatten: list[str] = ["Tech stack"]
details_cols_to_flatten: list[str] = ["Details"]

# Rozwijanie wskazanych kolumn i łączenie z oryginalnym DataFrame
tech_df = flatten_multiple_columns(tech_df, tech_cols_to_flatten)

# Wyodrębnienie kolumn językowych i usunięcie ich z tech_df
languages: list[str] = ["Polish", "English","French", "German"]
if all([lang in tech_df.columns for lang in languages]):
    lang_df: pd.DataFrame = tech_df[["Polish", "English", "French", "German"]]
    tech_df.drop(columns=["Polish", "English", "French", "German"], inplace=True)

# Rozwijanie szczegółów ofert
details_df = flatten_multiple_columns(details_df, details_cols_to_flatten)

# Szukanie i naprawa anomalii w nazewnictwie kolumn

In [14]:
if "Employment Type" in details_df.columns:
    details_df["Employment Type"] = details_df["Employment Type"].str.replace("B2B, Permanent", "Permanent, B2B")

# tech_cols jako lista nazw kolumn (umiejętności/technologii)
tech_cols_series: pd.Series = pd.Series(tech_df.columns)
tech_cols_series.to_csv("required_skills.csv", header = False, index = False)

# Stopień znajomości technologii - mapowanie poziomów na wartości liczbowe
technology_level: dict[str, int] = {         
    "-": 0,
    "nice to have": 1,
    "junior": 2,
    "regular": 3,
    "advanced": 4,
    "master": 5
}

# Poziom doświadczenia - mapowanie poziomów na wartości liczbowe
experience_level: dict[str, int] = {
    "-": 0,
    "Junior": 1,
    "Mid": 2,
    "Senior": 3,
    "C-level": 4,
}

def repair_col(row: pd.Series, cols: list[str]) -> str:
    """
    Zwraca nazwę pierwszej kolumny z listy 'cols', w której wartość w wierszu 'row' jest różna od "-".
    Jeśli żadna z kolumn nie spełnia warunku, zwraca "-".
    """
    return max(row[cols], key=lambda x: technology_level[x])

# Słownik tłumaczący polskie nazwy technologii na angielskie odpowiedniki oraz poprawiający literówki
polish2english: dict[str, str] = {
    "Biblioteki ML": "ML Libraries",
    "Bazy Danych": "Data Bases",
    "Rodo": "GDPR",
    "Kaban": "Kanban",
    "Regulatory requirements": "Regulatory Requirements",
    "data governance": "Data Governance",
    "Data Analystics": "Data Analytics",
    "product analytics tooling": "Product Analytics Tooling",
    "Azure Data stack": "Azure Data Stack",
    ", Prometheus": "Prometheus",
    "data algorithms": "Data Algorithms"
}

# Zastosowanie tłumaczeń i poprawek do nazw kolumn
tech_df.rename(columns=polish2english, inplace=True)


# Lista krotek z wariantami nazw tej samej technologii do ujednolicenia
to_unify = (
    ["Machine Learning", "Machine Learnign"],
    ["Google Cloud Platform", "GCP"],
    ["ETL", "ETL tools"],
    ["Probability and Statistics", "Statistics", "statystyka"],
    ["MS SQL Server", "MsSQL", "MS SQL", "Microsoft SQL", "SQL Server"],
    ["Data Warehousing", "Data wearhouse"],
    ["Amazon AWS", "AWS"],
    ["Power BI", "Microsoft BI", "BI tools", "Business Intelligence"],
    ["Data Lake", "Azure Data Lake Gen2", "Azure Data Lake"],
    ["Databases", "Database", "Data Bases"],
    ["Microsoft Azure", "Cloud", "Microsoft Azure Cloud", "Azure"],
    ["UNIX/LINUX", "Linux-Bash", "Linux server systems", "Unix"],
    ["Microsoft Office", "MS Office"],
    ["Git", "GitLab", "GitHub"],
    ["Confluence", "Jira/Confluence"],
    ["Apache Kafka", "Kafka"]
)

# Ujednolicanie kolumn na podstawie powyższych wariantów
for cols in to_unify:
    # Sprawdź, czy wszystkie warianty kolumn istnieją w DataFrame
    if all([x in tech_df.columns for x in cols]):
        # Wybierz najlepszą wartość z wariantów dla każdego wiersza
        repaired_col: pd.Series = tech_df.apply(lambda record: repair_col(record, cols), axis=1)
        # Usuń stare kolumny
        tech_df.drop(columns=cols, inplace=True)
        # Dodaj nową, ujednoliconą kolumnę pod poprawną nazwą (pierwszy wariant z listy)
        tech_df[cols[0]] = repaired_col

# Ponowne zastosowanie tłumaczeń i poprawek do nazw kolumn (na wypadek nowych kolumn po unifikacji)
tech_df.rename(columns=polish2english, inplace=True)

# Zapisywanie listy wymaganych umiejętności do CSV

In [15]:
tech_cols_series: pd.Series = pd.Series(tech_df.columns)
tech_cols_series.to_csv("required_skills.csv", header = False, index = False)

# Ujednolicanie poziomu znajomości języków

In [16]:
try:
    for col in lang_df.columns:
        print(col,lang_df[col].unique())

    # Zamiana wartości "advanced" na "C1"
    for col in lang_df.columns:
        lang_df.loc[:, col] = lang_df[col].replace("advanced", "C1")
        
except NameError:
    print("Brak ramki danych o umiejętnościach językowych")


# Zamiana wartości "advanced" na "C1"


Brak ramki danych o umiejętnościach językowych


# Zamiana typu kolumn na zmienną kategoryczną uporządkowaną

In [17]:
from pandas.api.types import CategoricalDtype

tech_levels_order = list(technology_level.keys())
cat_type = CategoricalDtype(categories=tech_levels_order, ordered=True)
for col in tech_df.columns:
    try:
        tech_df[col] = tech_df[col].astype(cat_type)
    except TypeError:
        continue

# Zapisywanie tematycznych ramek danych do plików csv

In [18]:
# --- Zapisywanie tematycznych ramek danych do plików csv ---
df_path = Path("dataframes")
df_path.mkdir(exist_ok=True)

csv_df_names = {
    "tech.csv": tech_df,
    "details.csv": details_df
}
try:
    csv_df_names["language.csv"] = lang_df
except NameError:
    pass

for csv_file_name, df in csv_df_names.items():
    df.to_csv(df_path / csv_file_name)